# 火焰检测模型剪枝
使用已经训练完成的网络，对网络进行结构化剪枝，使用yolo11n作为示例

In [ ]:
from ultralytics import YOLO
import torch
import copy

model_path = "../model/yolo11n.pt"
yolo = YOLO(model_path)
yolo.info()
yolo_copy =copy.deepcopy(yolo)
yolo_copy.val(data='fire_data.yaml', split="test", device=[0], imgsz=(384, 640))

当前模型在fasdd-test集上的mAP50为：

In [ ]:
print(yolo.model) # 打印模型结构

## 可剪枝层分析
我们仅仅对卷积层进行剪枝，分析YOLO11结构中可剪枝部分：    
1、跨模块：上层Conv连接下层Conv/C3k2.cv1.Conv  
2、模块内：C3k2中Bottleneck层中卷积层连接:Bottleneck.cv1.Conv与Bottleneck.cv2.Conv  
3、模块内：C3k2中多层Bottleneck层间的连接，如m.0.Bottleneck.cv2.Conv和m.1.Bottleneck.cv1  
4、模块内：C3k2中C3k中cv1.Conv和cv2.Conv与cv3.Conv中连接层  
5、模块内：C3k2层中存在split操作，cv1.Conv与cv2.Conv无法裁剪，可以将C3k2替换成C3k2_v2，后裁剪;C3k2_v2可裁剪cv0.Conv、cv1.Conv与cv2.Conv连接层  
6、跨模块：上层C3k2中的输出卷积层C3k2.cv2.Conv连接下层卷积层Conv  
7、模块内：SPPF内部cv1输出与cv2连接部分  
8、模块内：Detect头class/Bbox，紧邻内部conv分支  

In [ ]:
from ultralytics.nn.modules import Bottleneck, Conv, C2f, C3k2, SPPF, Detect
import torch
from torch.nn.modules.container import Sequential
import copy
import os

class Pruner:
    """
    原生 YOLO 模型网络剪枝器。
    这套实现抛弃了第三方库，纯原生使用权重 slicing 来手动剪减张量。
    内建了如针对残差Bottleneck结构的保护等专业功能。
    """
    def __init__(self):
        pass

    def get_keep_indices(self, conv: Conv, keeping_rate: float, mode='L2'):
        """基于 L2 范数或 BN 的 Gamma 值做重要性判断并返回保留索引"""
        if mode == 'gamma':
            gamma = conv.bn.weight.data.detach()
            channels = len(gamma)
            keep_channels = max(1, int(channels * keeping_rate))
            _, top_inds = torch.topk(gamma.abs(), k=keep_channels)
        elif mode == 'L2':
            weight = conv.conv.weight.data.detach()
            norms_sq = (weight ** 2).sum(dim=[1, 2, 3])
            channels = len(norms_sq)
            keep_channels = max(1, int(channels * keeping_rate))
            _, indices = torch.sort(norms_sq, descending=True)
            top_inds = indices[:keep_channels]
        else:
            raise ValueError("mode 必须是 'L2' 或 'gamma'")

        print(f"[*] 通道评估 => 原: {channels} 期望保留: {keep_channels} (比率: {keeping_rate:.2f})")
        return top_inds.tolist(), keep_channels

    def prune_conv_pair(self, conv1: Conv, conv2: Conv, keeping_rate: float, mode='L2'):
        """
        单向标准剪枝：剪掉 conv1 输出和 conv2 输入对应的同等索引通道。
        适用于上下游没有受到 Add/Concat 的硬依赖时。
        """
        keep_idxs, keep_channels = self.get_keep_indices(conv1, keeping_rate, mode)

        # 处理第一层的输出端相关权重(Batch Norm + 卷积)
        conv1.bn.weight.data = conv1.bn.weight.data[keep_idxs]
        conv1.bn.bias.data = conv1.bn.bias.data[keep_idxs]
        conv1.bn.running_var.data = conv1.bn.running_var.data[keep_idxs]
        conv1.bn.running_mean.data = conv1.bn.running_mean.data[keep_idxs]
        conv1.bn.num_features = keep_channels

        conv1.conv.weight.data = conv1.conv.weight.data[keep_idxs]
        if conv1.conv.bias is not None:
            conv1.conv.bias.data = conv1.conv.bias.data[keep_idxs]
        conv1.conv.out_channels = keep_channels

        # 处理第二层的输入端映射
        conv2.conv.in_channels = keep_channels
        conv2.conv.weight.data = conv2.conv.weight.data[:, keep_idxs]
        if conv2.conv.bias is not None:
            # bias of conv2 normally isn't related to in_channels, but keep as is natively
            pass

pruner = Pruner()


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

def plot_sensitivity_curve(sparsities, accuracies):
    prune_ratio = [1-x for x in sparsities]

    # 裁剪前的准确率，即未剪枝时的准确率，替换为您实际跑出来的准确率
    base_accuracy = 0.624

    # 创建图形
    plt.figure(figsize=(10, 6))

    # 绘制剪枝率与准确率的关系折线图
    plt.plot(prune_ratio, accuracies, 'o-', color='#1f77b4', linewidth=2, markersize=8, label='accuracy after prune')

    # 绘制裁剪前的准确率基准线（红色水平线）
    plt.axhline(y=base_accuracy, color='r', linestyle='--', linewidth=2, label='base_accuracy')

    # 添加数据标签
    for i, acc in enumerate(accuracies):
        plt.annotate(f'{acc:.2f}', (prune_ratio[i], accuracies[i]),
                     textcoords="offset points", xytext=(0,10), ha='center')

    # 设置图标标题和标签
    plt.title('Sensitivity Curves: Validation Accuracy vs. Pruning Sparsity', fontsize=14)
    plt.xlabel('pruning ratio', fontsize=12)
    plt.ylabel('mAP50', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)

    # 设置x轴范围
    plt.xlim(-0.05, max(prune_ratio) * 1.1)
    plt.ylim(0, max(accuracies) * 1.2)

    # 添加图例
    plt.legend(loc='lower left')

    # 显示图片
    plt.tight_layout()
    plt.show()

def plot_multi_sensitivity_curve(sparsities, accuracies, sub_curves_name):
    base_accuracy = 0.624
    prune_ratio = [1-x for x in sparsities]
    sub_line_count = len(accuracies)

    # 创建图形
    plt.figure(figsize=(12, 8))

    # 为每层绘制曲线 （使用不同的颜色和线型）
    colors = plt.cm.tab10(np.linspace(0, 1, sub_line_count))
    line_styles = ['-', '--', '-.', ':'] # 线型

    # 绘制裁剪前的准确率基准线（红色水平线）
    plt.axhline(y=base_accuracy, color='r', linestyle='--', linewidth=2, label='base_accuracy')

    # 添加数据标签
    for i in range(sub_line_count):
        for j, acc in enumerate(accuracies[i]):
            plt.annotate(f'{acc:.2f}',
                         (prune_ratio[i], accuracies[i][j]),
                         textcoords='offset points',
                         xytext=(0, 10),
                         ha='center',
                         fontsize=9)

    # 设置图标标题和标签
    plt.title('Sensitivity Curves: Validation Accuracy vs. Pruning Sparsity', fontsize=16, pad=20)
    plt.xlabel('pruning ratio', fontsize=14, labelpad=10)
    plt.ylabel('mAP50', fontsize=14, labelpad=10)
    plt.grid(True, linestyle='--', alpha=0.7)

    # 设置x轴范围
    plt.xlim(-0.05, max(prune_ratio) * 1.1)
    plt.ylim(0, max([max(acc) for acc in accuracies]) * 1.2)

    # 添加图例
    plt.legend(loc='upper right', bbox_to_anchor=(1.0, 1.0), ncol=2, fontsize=10, frameon=True)

    # 显示图片
    plt.tight_layout(rect=[0, 0, 0.85, 1]) #为图例留出空间
    plt.show()

## 不同裁剪方案对比（gamma vs L2）

# 一、network_slimming
network_slimming技术：使用bn层Gamma因子对channel进行排序，裁掉重要性低的通道
裁掉第0层的卷积层，看看精度下降多少

In [ ]:
gamma_accuracies = []
sparsities = [0.875, 0.75, 0.5, 0.25]
for i, sparsity in enumerate(sparsities):
    yolo_copy = copy.deepcopy(yolo)
    model_seq = yolo_copy.model.model
    pruner.prune_conv_pair(model_seq[0], model_seq[1], sparsity, mode='gamma')
    result = yolo_copy.val(data='fire_data.yaml', split='test', device=[0])
    gamma_accuracies.append(result.box.map)

## 二、L2范数裁剪
使用卷积核的L2范数确定裁剪通道，同样使用第0个通道作实验

In [ ]:
l2_accuracies = []
sparsities = [0.875, 0.75, 0.5, 0.25]
for i, sparsity in enumerate(sparsities):
    yolo_copy = copy.deepcopy(yolo)
    seq = yolo_copy.model.model
    pruner.prune_conv_pair(seq[0], seq[1], sparsity, "L2")
    result = yolo_copy.val(data='fire_data.yaml', split='test', device=[0])
    l2_accuracies.append(result.box.map)

## 三、方案对比与原因分析
对比两种不同裁剪方案：仅仅对第0层

In [ ]:
compare_accuracies = []
compare_accuracies.append(gamma_accuracies)
compare_accuracies.append(l2_accuracies)
plot_multi_sensitivity_curve(sparsities, compare_accuracies, ["gamma prune", "l2 prune"])

对于我们网络的第零层，对于gamma因子和L2范数裁剪，L2范数裁剪明显优于gamma因子裁剪，我们可以分析一下网络卷积层gamma因子分布与weight分布对比

In [ ]:
yolo_copy = copy.deepcopy(yolo)

conv_weight = []
bn_weight = []

# 遍历模型参数
for name, param in yolo_copy.model.model.named_parameters():
    # 提取卷积层权重
    if 'conv' in name and 'weight' in name:
        weights = param.detach().view(-1).cpu()
        conv_weight.append(weights)

    # 提取BN层gamma(BatchNorm2d weight)
    elif 'bn' in name and 'weight' in name:
        gamma = param.detach().view(-1).cpu()
        bn_gammas.append(gamma)

# 合并所有权重
conv_weights = torch.cat(conv_weights).numpy()
bn_gammas = torch.cat(bn_gammas).numpy()

# 创建两个子图
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 卷积层权重分布
axes[0].hist(conv_weights, bins=256, density=True,
             color='blue', alpha=0.7, edgecolor='black')
axes[0].set_title('Convolutional Later Weights Distribution', fontsize=14)
axes[0].set_xlabel("Weight Value", fontsize=12)
axes[0].set_ylabel("Density", fontsize=12)
axes[0].grid(True, linestyle='--', alpha=0.7)

# BN层gamma分布
axes[0].hist(bn_gammas, bins=256, density=True,
             color='blue', alpha=0.7, edgecolor='black')
axes[0].set_title('Convolutional Later Weights Distribution', fontsize=14)
axes[0].set_xlabel("Weight Value", fontsize=12)
axes[0].set_ylabel("Density", fontsize=12)
axes[0].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.suptitle('Weight and Bn gamma Distribution Analysis', fontsize=16, y=1.02)
plt.show()

观察网络中BN层gamma的分布和所有卷积层中weight的分布，可以看出，weight大部分是集中在0附近的，这个与我们的正则化训练有关。同时由于weight这种0附近正态分布特性，有很多冗余weight，在weight为0时，该weight对后续层的推理结果贡献很小，是我们裁剪的原理。gamma则没有这种集中在0附近的分布，所以在我们Gamma的L1裁剪不如weight的L2裁剪效果，当然我们可以遵循network slimming原论文方法进行L1稀疏化训练来改变Gamma分布使其在0附近有一个尖峰，来提升L1裁剪的效果

## C3k2中bottelneck层裁剪
以C3k2中bottelneck层为例，其他层可以仿造该代码来裁剪

In [ ]:
c3k2_accuracies = []
sparsities = [0.875, 0.75, 0.5, 0.25]
for i in [2, 4, 6, 8]:
    accuracy = []
    for _, sparsity in enumerate(sparsities):
        yolo_copy = copy.deepcopy(yolo)
        seq = yolo_copy.model.model
        for name, m in seq[i].named_modules():
            if isinstance(m, Bottleneck):
                pruner.prune_conv_pair(m.cv1, m.cv2, sparsity, "L2")
        result = yolo_copy.val(data='fire_data.yaml', split='test', device=[0])
        accuracy.append(result.box.map)
    c3k2_accuracies.append(accuracy)

In [ ]:
plot_multi_sensitivity_curve(sparsities, c3k2_accuracies, ["module2_bottleneck", "module4_bottleneck", "module6_bottleneck", "module8_bottleneck"])

对每个可裁剪层做类似敏感度分析，然后给出每层最佳剪枝比例（如：精度掉点不超过百分之5），使用pruner对每层进行剪枝，剪枝完成后进行多轮重训

## 四、多轮全局结构化剪枝与微调重训 (Fine-tuning)
前面基于各层的敏感度评估，可以得出哪些层能以何种比例丢弃。现在，我们将其应用于整个网络，然后重启训练循环，这是保证模型掉点后能够通过重新反向传播训练弥补损失的**必不可少**环节。

In [ ]:
# 1. 定义多轮迭代剪枝方案并对全网批量执行
print('========== 启动全局底层自动裁剪 ==========')
yolo_pruned = copy.deepcopy(yolo)
seq = yolo_pruned.model.model
prune_mgr = Pruner()

# 我们选定基于敏感度表现不错的阈值，可裁剪层每层选择一个阈值，下面的阈值选取仅供参考

# backbone剪枝
# backbone conv与conv连接剪枝
prune_mgr.prune_conv_pair(seq[0].Conv, seq[1].Conv, keeping_rate=0.875)
# backbone conv与C3k2连接剪枝
for i in [1, 3, 5, 7]:
    target_sparsity = [0.875, 0.75, 0.5, 0.5]
    prune_mgr.prune_conv_pair(seq[i].Conv, seq[i+1].cv1, target_sparsity[i])
# backbone c3k2与conv连接剪枝
for i in [2, 8]:
    target_sparsity = [0.875, 0.5]
    prune_mgr.prune_conv_pair(seq[i].cv2, seq[i+1].Conv, target_sparsity[i])
# backbone c3k2 bottelneck层剪枝
for i in range(10):
    for name, m in seq[i].named_modules
    if isinstance(m, Bottleneck):
        prune_mgr.prune_conv_pair(m.cv1, m.cv2, 0.5)

# neck剪枝
# neck c3k2 bottleneck层剪枝
for i in [13, 16, 19, 22]:
    for name, m in seq[i].named_modules
    if isinstance(m, Bottleneck):
        prune_mgr.prune_conv_pair(m.cv1, m.cv2, 0.5)

# head剪枝
for i in range(3):
    prune_mgr.prune_conv_pair(seq[23].cv2[i][0], seq[23].cv2[i][1], 0.875)
print('========== 底层网络压缩完成 ==========')


In [ ]:
# 2. 即刻验证因剪枝产生的维度是否能够顺利运行前向推理 (验证阶段会发现掉点现象是正常的)
print('验证模型连接和初步精度...')
res = yolo_pruned.val(data='fire_data.yaml', split='test', device=[0])
print(f'剪裁后未经重训直接推断的 MAP50: {res.box.map50:.5f}')

In [ ]:
# 3. 运行模型的重新微调训练 (Finetuning) 来找回精度
# 提示：如果是极度压缩的多轮剪枝(Iterative Pruning)，需要将 剪枝 + 训练 套在一个对于 target_sparsity 梯度逐步递减的 for 热启动大循环外圈中。这里提供单轮标准微调作为核心示范：
# 不使用yolo_pruned.train(...), 而是手动构建trainer并绕过模型重新构造
print('========== 开始重训微调阶段 ==========')
from ultralytics.models.yolo.detect.train import DetectionTrainer
args = dict(
    model='../model/yolo11n.pt',
    data='fire_data.yaml',
    epochs=25,
    lr0=0.001,
    batch=16,
    device=[0],
    name='fire_pruned_finetune',
)
trainer = DetectionTrainer(overrides=args)
# 将内存中已经剪枝好的模型 DetectionModel(其嵌套在YOLO().model.model中) 传递过去
trainer.model = yolo_pruned.model
trainer.train()

# 训练完成后保存
os.makedirs('../model/pruned_target', exist_ok=True)
yolo_pruned.save('../model/pruned_target/final_pruned.pt')
print('========== 完全交付：微调网络已成功保存 ==========')